In [1]:
import pymrio
import pandas as pd
import os

# Load core IO system
exio = pymrio.parse_exiobase3(
    path='D:/Programming/databases/IOT_2022_pxp'
)

# Just load the physical data - NO calc_system needed for exploration!
data_path = 'D:/Programming/databases/IOT_2022_pxp'

In [2]:
plastic_demand_kg = pd.DataFrame(
    columns=["SE", "CN"],
    index=pd.MultiIndex.from_tuples(
        [
            ("Plastic_A", "kg"),
            ("Plastic_B", "kg"),
        ]
    ),
    data=[
        [200, 200],
        [50, 50],
    ],
)

plastic_demand_kg.index.names = ["product", "unit"]
plastic_demand_kg.columns.names = ["region"]
plastic_demand_kg

,region,SE,CN
product,unit,,
Plastic_A,kg,200,200
Plastic_B,kg,50,50


In [3]:
plastic_map_to_million_eur = pd.DataFrame(
    columns=[
        "product",
        "unit",
        "product__product",
        "unit__unit",
        "factor",
    ],
    data=[
        # SE
        ["Plastic_A", "kg", "Plastics, basic", "million EUR", 5.0 / 1000000],
        ["Plastic_B", "kg", "Plastics, basic", "million EUR", 2.0 / 1000000],
        # CN (50% of SE prices)
        ["Plastic_A", "kg", "Plastics, basic", "million EUR", 2.5 / 1000000],
        ["Plastic_B", "kg", "Plastics, basic", "million EUR", 1.0 / 1000000],
    ],
)

plastic_map_to_million_eur

,product,unit,product__product,unit__unit,factor
0,Plastic_A,kg,"Plastics, basic",million EUR,0.000005
1,Plastic_B,kg,"Plastics, basic",million EUR,0.000002
2,Plastic_A,kg,"Plastics, basic",million EUR,0.000003
3,Plastic_B,kg,"Plastics, basic",million EUR,0.000001


In [4]:
# SE
plastic_fd_SE = pymrio.convert(
    plastic_demand_kg[["SE"]],
    plastic_map_to_million_eur.iloc[[0, 1]],
)

# CN
plastic_fd_CN = pymrio.convert(
    plastic_demand_kg[["CN"]],
    plastic_map_to_million_eur.iloc[[2, 3]],
)
plastic_final_demand = pd.concat(
    [plastic_fd_SE, plastic_fd_CN],
    axis=1
)

plastic_final_demand

,region,SE,CN
product,unit,,
"Plastics, basic",million EUR,0.0011,0.00055


In [ ]:
%%%%%%%%%%%%%%

In [ ]:
exio.calc_all()

In [ ]:
type(exio.extensions)

In [ ]:
exio.extensions

In [ ]:
print(exio.air_emissions.DataFrames)

In [ ]:
type(air_emissions)

In [ ]:
y = pd.Series(
    data=0.0,
    index=exio.A.index,
    name="final_demand"
)

product_name = "Plastics, basic"

y.loc[("SE", product_name)] = plastic_final_demand.loc[
    ("Plastics, basic", "million EUR"), "SE"
]

y.loc[("CN", product_name)] = plastic_final_demand.loc[
    ("Plastics, basic", "million EUR"), "CN"
]

y

In [ ]:
df_Y = exio.Y

pymrio.index_fullmatch(df_Y, region ="SE", sector="Plastics, basic")

In [ ]:
x = exio.L.dot(y)


In [ ]:
gwp = exio.extensions["air emissions"]

gwp_result = gwp.L.dot(y)

total_gwp = gwp_result.sum()
total_gwp

In [ ]:
air_emissions = getattr(exio, "air_emissions")

In [ ]:
air_emissions = getattr(exio, "air_emissions")

ghg_result = air_emissions.L.dot(y)

total_ghg = ghg_result.sum().sum()
print("Total air emissions:", total_ghg)

In [ ]:
ghg_result = exio.air_emissions.S_Y.dot(y)

In [ ]:
# Consumer-based footprint by region
regional_ghg = exio.air_emissions.D_cba_reg

# If you want global total:
total_ghg = regional_ghg.sum().sum()
print("Total GHG footprint:", total_ghg)
